In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import REGISTRY_COUNTS_CSV, PLATFORM_COUNTS_CSV, ANALYSIS_DIR

sns.set_theme(style="whitegrid")

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_registry = pd.read_csv(REGISTRY_COUNTS_CSV)
print(f"Registry shape: {df_registry.shape}")
df_registry.head()

In [ ]:
df_platform = pd.read_csv(PLATFORM_COUNTS_CSV)
print(f"Platform shape: {df_platform.shape}")
df_platform.head()

In [ ]:
print("=== Registry ===")
df_registry.info()
print(f"\nNull values:\n{df_registry.isna().sum()}")
print(f"\n=== Platform ===")
df_platform.info()
print(f"\nNull values:\n{df_platform.isna().sum()}")

In [ ]:
df = df_registry.merge(
    df_platform[["province_abbr", "platform_entities"]],
    on="province_abbr",
    how="left",
)

df["platform_entities"] = df["platform_entities"].fillna(0).astype(int)
df["coverage_gap"] = df["entities_total"] - df["platform_entities"]
df["coverage_ratio"] = df["platform_entities"] / df["entities_total"]

print(f"Merged shape: {df.shape}")
df.head(10)

In [ ]:
df[["entities_total", "platform_entities", "coverage_gap", "coverage_ratio"]].describe()

In [ ]:
df_region = (
    df.groupby("region_name", as_index=False)
    .agg(
        registry_total=("entities_total", "sum"),
        platform_total=("platform_entities", "sum"),
    )
)

df_region["coverage_ratio"] = df_region["platform_total"] / df_region["registry_total"]
df_region["coverage_gap"] = df_region["registry_total"] - df_region["platform_total"]
df_region = df_region.sort_values("coverage_gap", ascending=False)

df_region

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

df_plot = df_region.sort_values("coverage_gap", ascending=True)
ax.barh(df_plot["region_name"], df_plot["coverage_gap"])

ax.set_xlabel("Coverage Gap (entities)")
ax.set_title("Coverage Gap by Region")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "coverage_gap_by_region.png", dpi=150)
plt.show()

In [ ]:
pivot = df.pivot_table(
    index="region_name",
    columns="province_abbr",
    values="coverage_ratio",
    fill_value=0,
)

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.5, ax=ax, fmt=".2f", annot=True)

ax.set_title("Coverage Ratio by Province")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "coverage_gap_heatmap.png", dpi=150)
plt.show()

In [ ]:
top_gap = df.sort_values("coverage_gap", ascending=False).head(15)
top_gap[["region_name", "province_name", "province_abbr", "entities_total", "platform_entities", "coverage_gap", "coverage_ratio"]]

In [ ]:
output_path = ANALYSIS_DIR / "coverage_gap_by_province.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

## Key Insights

1. Most provinces have **zero platform coverage**, indicating significant expansion potential across Italy.
2. The largest coverage gaps are concentrated in the most populated regions (Lombardia, Lazio, Veneto, Emilia-Romagna).
3. The platform currently covers only **one province per region** (the regional capital), leaving all other provinces unserved.
4. Even in covered provinces, the **coverage ratio remains low**, suggesting room for deeper penetration alongside geographic expansion.